# Notebook 04: 1D CNN Deep Learning Baseline

**Objective**: Train and evaluate the temporal 1D Convolutional Neural Network (`OpportunityCNN`) across all 133 on-body wearable channels.

### Hardware Requirement:
- `EXPERIMENT_MODE = 'DEBUG'`: Runs a fast 1-epoch smoke test on CPU or GPU for instant verification.
- `EXPERIMENT_MODE = 'FAST'` or `'FULL'`: **CUDA GPU REQUIRED** (e.g. Google Colab T4 GPU). Prevents laptop CPU freezing.

## 1. Configure Experiment Mode

In [ ]:
# Choose: 'DEBUG' (1 epoch test), 'FAST' (5 epochs GPU), or 'FULL' (50 epochs early stopping GPU)
EXPERIMENT_MODE = "DEBUG"

import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt

current = Path.cwd()
REPO_ROOT = current.parent if current.name == "notebooks" else current
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import src.api as api
env_info = api.setup_project()

## 2. Load Processed Sliding Windows & Prepare DataLoaders

In [ ]:
subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
X_train, y_train = subset["X_train"], subset["y_train"]
X_val, y_val = subset["X_val"], subset["y_val"]
X_test, y_test = subset["X_test"], subset["y_test"]

if EXPERIMENT_MODE == "DEBUG":
    # Select minimal subset for fast CPU smoke test
    X_train, y_train = X_train[:64], y_train[:64]
    X_val, y_val = X_val[:32], y_val[:32]
    X_test, y_test = X_test[:32], y_test[:32]

train_loader, val_loader, test_loader = api.create_dataloaders(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    batch_size=64 if EXPERIMENT_MODE != "DEBUG" else 16,
)
print(f"DataLoaders Ready: {len(train_loader)} train batches, {len(val_loader)} val batches.")

## 3. Instantiate 1D CNN Architecture

In [ ]:
cnn_model = api.build_cnn(
    in_channels=133,
    num_classes=5,
    sequence_length=30,
    dropout=0.3
)

total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"OpportunityCNN Model Initialized! Trainable parameters: {total_params:,}")

## 4. Train Model (Automatic GPU/CUDA Guard)

In [ ]:
best_cnn, history = api.train_model(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    mode=EXPERIMENT_MODE,
    lr=1e-3,
    experiment_name="opportunity_cnn"
)
print("Training Complete!")

## 5. Evaluate on Held-Out Test Set

In [ ]:
test_metrics = api.evaluate_model(best_cnn, test_loader)

print("\n" + "=" * 50)
print(" 1D CNN TEST EVALUATION")
print("=" * 50)
print(f"Accuracy    : {test_metrics['accuracy'] * 100:.2f}%")
print(f"Macro F1    : {test_metrics['macro_f1'] * 100:.2f}%")
print(f"Weighted F1 : {test_metrics['weighted_f1'] * 100:.2f}%")
print("=" * 50)